# Sinh toàn bộ hình ảnh cho báo cáo luận văn

**Đề tài:** Dự đoán và khuyến nghị kết quả học tập bằng kiến trúc lai CNN-BiLSTM + Context MLP.

Notebook này đọc trực tiếp các tệp dữ liệu và kết quả thực nghiệm trong dự án, kiểm tra lại metric từ dự đoán khóa kiểm thử, rồi sinh toàn bộ sơ đồ và biểu đồ ở độ phân giải cao. Không có metric đánh giá nào được nhập thủ công.

Đầu ra:

- `reports/final/figures/notebook_all/*.png`
- `reports/final/figures/notebook_all/figure_manifest.csv`
- `reports/final/figures/notebook_all/figure_manifest.json`
- `reports/final/bo_hinh_anh_luan_van_tu_du_lieu_that.zip`

In [2]:
from __future__ import annotations

import json
import math
import re
import textwrap
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw, ImageFont


def locate_project_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent, Path(r"C:\Huflit\kltn")]
    for candidate in candidates:
        if (candidate / "src" / "config.py").exists() and (candidate / "reports" / "final").exists():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy thư mục gốc của dự án.")


ROOT = locate_project_root()
OUT = ROOT / "reports" / "final" / "figures" / "notebook_all"
OUT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ROOT / "reports" / "final" / "bo_hinh_anh_luan_van_tu_du_lieu_that.zip"

DATASETS = {
    "student-mat": {
        "label": "Student-Mat",
        "target": "G3",
        "kind": "student",
        "raw": ROOT / "data" / "raw" / "student-mat.csv",
        "sep": ";",
    },
    "student-por": {
        "label": "Student-Por",
        "target": "G3",
        "kind": "student",
        "raw": ROOT / "data" / "raw" / "student-por.csv",
        "sep": ";",
    },
    "xapi": {
        "label": "xAPI",
        "target": "Class",
        "kind": "xapi",
        "raw": ROOT / "data" / "raw" / "xAPI-Edu-Data.csv",
        "sep": ",",
    },
}

CLASS_NAMES = ["Low", "Medium", "High"]
NAVY = "#17365D"
BLUE = "#4472C4"
SKY = "#D9EAF7"
PALE = "#F3F6FA"
GREEN = "#70AD47"
PALE_GREEN = "#E2F0D9"
GOLD = "#FFC000"
PALE_GOLD = "#FFF2CC"
RED = "#C0504D"
PALE_RED = "#F4CCCC"
GRAY = "#666666"
LIGHT_GRAY = "#E7E6E6"
GRID = "#D9E1F2"
WHITE = "#FFFFFF"
BLACK = "#111111"
COLORS = [BLUE, GREEN, GOLD, RED, "#8064A2", "#4BACC6"]

FIGURE_MANIFEST = []
print("Project root:", ROOT)
print("Output folder:", OUT)

Project root: C:\Huflit\kltn
Output folder: C:\Huflit\kltn\reports\final\figures\notebook_all


## 1. Đọc dữ liệu và kiểm tra khoa học

Các metric Accuracy, Precision-Macro, Recall-Macro, F1-Macro, RMSE và R² được tính lại từ `True_Label` và `Pred_Label`. Notebook dừng bằng `AssertionError` nếu kết quả không khớp JSON locked-test.

In [3]:
def confusion_matrix_3(y_true, y_pred):
    cm = np.zeros((3, 3), dtype=int)
    for truth, pred in zip(np.asarray(y_true, dtype=int), np.asarray(y_pred, dtype=int)):
        cm[truth, pred] += 1
    return cm


def per_class_metrics(cm):
    rows = []
    for cls in range(3):
        tp = float(cm[cls, cls])
        fp = float(cm[:, cls].sum() - tp)
        fn = float(cm[cls, :].sum() - tp)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        rows.append({"Class": CLASS_NAMES[cls], "Precision": precision, "Recall": recall, "F1": f1, "Support": int(cm[cls].sum())})
    return pd.DataFrame(rows)


def recompute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    cm = confusion_matrix_3(y_true, y_pred)
    per_class = per_class_metrics(cm)
    accuracy = float(np.mean(y_true == y_pred))
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    denominator = float(np.sum((y_true - y_true.mean()) ** 2))
    r2 = float(1.0 - np.sum((y_true - y_pred) ** 2) / denominator)
    metrics = {
        "Accuracy": accuracy,
        "F1-Macro": float(per_class["F1"].mean()),
        "Precision-Macro": float(per_class["Precision"].mean()),
        "Recall-Macro": float(per_class["Recall"].mean()),
        "RMSE": rmse,
        "R2": r2,
    }
    return metrics, cm, per_class


def parse_final_report(path: Path):
    text = path.read_text(encoding="utf-8")
    cv_match = re.search(r"Optuna Best CV F1:\s*([0-9.]+)", text)
    params_match = re.search(r"Best Params:\s*(\{.*?\})\s*\n\nFinal", text, re.S)
    if not cv_match or not params_match:
        raise ValueError(f"Không phân tích được báo cáo: {path}")
    return float(cv_match.group(1)), json.loads(params_match.group(1))


loaded = {}
validation_rows = []

for dataset, config in DATASETS.items():
    metrics_path = ROOT / "reports" / "final" / "metrics" / f"{dataset}_3class_locked_test_metrics.json"
    predictions_path = ROOT / "reports" / "final" / "predictions" / f"{dataset}_3class_predictions.csv"
    importance_path = ROOT / "reports" / "final" / "explanations" / f"{dataset}_3class_feature_importance.csv"
    recommendations_path = ROOT / "reports" / "final" / "recommendations" / f"{dataset}_3class_learning_paths.csv"
    train_path = ROOT / "data" / "processed" / "final" / f"{dataset}_3class_train_pool.csv"
    test_path = ROOT / "data" / "processed" / "final" / f"{dataset}_3class_locked_test.csv"
    report_path = ROOT / "reports" / "final" / f"{dataset}_3class_final_report.txt"

    official = json.loads(metrics_path.read_text(encoding="utf-8"))
    predictions = pd.read_csv(predictions_path)
    importance = pd.read_csv(importance_path).sort_values("Importance", ascending=False).reset_index(drop=True)
    recommendations = pd.read_csv(recommendations_path)
    train_pool = pd.read_csv(train_path)
    locked_test = pd.read_csv(test_path)
    raw = pd.read_csv(config["raw"], sep=config["sep"])
    cv_f1, best_params = parse_final_report(report_path)

    recalculated, cm, per_class = recompute_metrics(predictions["True_Label"], predictions["Pred_Label"])
    if len(predictions) != len(locked_test):
        raise AssertionError(f"{dataset}: số dự đoán không bằng số mẫu locked test")
    for metric, official_value in official.items():
        difference = abs(float(official_value) - float(recalculated[metric]))
        if difference > 1e-9:
            raise AssertionError(f"{dataset}: {metric} lệch {difference}")
        validation_rows.append({
            "Dataset": config["label"], "Metric": metric,
            "JSON": float(official_value), "Recomputed": float(recalculated[metric]),
            "Absolute difference": difference,
        })

    loaded[dataset] = {
        "config": config,
        "metrics": official,
        "predictions": predictions,
        "importance": importance,
        "recommendations": recommendations,
        "train": train_pool,
        "test": locked_test,
        "raw": raw,
        "cv_f1": cv_f1,
        "best_params": best_params,
        "cm": cm,
        "per_class": per_class,
        "sources": {
            "metrics": metrics_path, "predictions": predictions_path,
            "importance": importance_path, "recommendations": recommendations_path,
            "train": train_path, "test": test_path, "raw": config["raw"],
            "report": report_path,
        },
    }

validation_table = pd.DataFrame(validation_rows)
print(validation_table.to_string(index=False))
print("\nĐã xác minh toàn bộ metric locked-test khớp dữ liệu dự đoán.")

    Dataset          Metric     JSON  Recomputed  Absolute difference
Student-Mat        Accuracy 0.886076    0.886076         0.000000e+00
Student-Mat        F1-Macro 0.890504    0.890504         0.000000e+00
Student-Mat Precision-Macro 0.878136    0.878136         0.000000e+00
Student-Mat    Recall-Macro 0.917004    0.917004         0.000000e+00
Student-Mat            RMSE 0.337526    0.337526         0.000000e+00
Student-Mat              R2 0.771969    0.771969         0.000000e+00
Student-Por        Accuracy 0.861538    0.861538         0.000000e+00
Student-Por        F1-Macro 0.839355    0.839355         0.000000e+00
Student-Por Precision-Macro 0.811042    0.811042         0.000000e+00
Student-Por    Recall-Macro 0.881624    0.881624         0.000000e+00
Student-Por            RMSE 0.372104    0.372104         0.000000e+00
Student-Por              R2 0.606326    0.606326         0.000000e+00
       xAPI        Accuracy 0.760417    0.760417         0.000000e+00
       xAPI        F

## 2. Hàm vẽ dùng Pillow

Notebook dùng Pillow thay vì phụ thuộc Matplotlib, nhờ đó có thể chạy ngay trong môi trường dự án hiện tại. Mỗi ảnh được lưu ở 220 DPI và được ghi vào manifest cùng danh sách tệp nguồn.

In [4]:
def font(size: int, bold: bool = False):
    windows_fonts = Path(r"C:\Windows\Fonts")
    candidates = [windows_fonts / ("timesbd.ttf" if bold else "times.ttf"), windows_fonts / ("arialbd.ttf" if bold else "arial.ttf")]
    for candidate in candidates:
        if candidate.exists():
            return ImageFont.truetype(str(candidate), size)
    return ImageFont.load_default()


def new_image(width=1800, height=1050):
    image = Image.new("RGB", (width, height), WHITE)
    return image, ImageDraw.Draw(image)


def fit_text(text, width_px, size, bold=False):
    approximate_chars = max(8, int(width_px / max(size * 0.52, 1)))
    return textwrap.wrap(str(text), width=approximate_chars, break_long_words=False) or [""]


def draw_centered(draw, box, text, size=34, bold=False, fill=BLACK, spacing=7):
    x1, y1, x2, y2 = box
    lines = []
    for paragraph in str(text).split("\n"):
        lines.extend(fit_text(paragraph, x2 - x1, size, bold))
    fnt = font(size, bold)
    line_heights = [draw.textbbox((0, 0), line, font=fnt)[3] - draw.textbbox((0, 0), line, font=fnt)[1] for line in lines]
    total_height = sum(line_heights) + spacing * max(0, len(lines) - 1)
    y = y1 + (y2 - y1 - total_height) / 2
    for line, line_height in zip(lines, line_heights):
        bbox = draw.textbbox((0, 0), line, font=fnt)
        x = x1 + (x2 - x1 - (bbox[2] - bbox[0])) / 2
        draw.text((x, y), line, font=fnt, fill=fill)
        y += line_height + spacing


def draw_title(draw, text, subtitle=None, width=1800):
    draw_centered(draw, (45, 18, width - 45, 86), text, 44, True, NAVY)
    if subtitle:
        draw_centered(draw, (65, 82, width - 65, 128), subtitle, 24, False, GRAY)


def draw_box(draw, xy, heading, body="", fill=SKY, outline=NAVY, heading_size=31, body_size=25):
    draw.rounded_rectangle(xy, radius=18, fill=fill, outline=outline, width=4)
    x1, y1, x2, y2 = xy
    draw_centered(draw, (x1 + 10, y1 + 8, x2 - 10, y1 + 66), heading, heading_size, True, NAVY)
    if body:
        draw_centered(draw, (x1 + 16, y1 + 70, x2 - 16, y2 - 10), body, body_size, False, BLACK)


def draw_arrow(draw, start, end, color=NAVY, width=5):
    draw.line([start, end], fill=color, width=width)
    angle = math.atan2(end[1] - start[1], end[0] - start[0])
    length = 21
    left = (end[0] - length * math.cos(angle - 0.55), end[1] - length * math.sin(angle - 0.55))
    right = (end[0] - length * math.cos(angle + 0.55), end[1] - length * math.sin(angle + 0.55))
    draw.polygon([end, left, right], fill=color)


def add_manifest(filename, title, category, sources):
    source_strings = [str(Path(source).resolve().relative_to(ROOT)) if Path(source).resolve().is_relative_to(ROOT) else str(source) for source in sources]
    FIGURE_MANIFEST.append({
        "filename": filename,
        "title": title,
        "category": category,
        "data_sources": " | ".join(source_strings),
    })


def save_image(image, filename, title, category, sources):
    path = OUT / filename
    image.save(path, dpi=(220, 220), optimize=True)
    add_manifest(filename, title, category, sources)
    return path


def draw_axes(draw, plot, ymin, ymax, ticks=5, percent=False):
    x1, y1, x2, y2 = plot
    draw.line((x1, y1, x1, y2), fill=BLACK, width=3)
    draw.line((x1, y2, x2, y2), fill=BLACK, width=3)
    for idx in range(ticks + 1):
        value = ymin + (ymax - ymin) * idx / ticks
        y = y2 - (y2 - y1) * idx / ticks
        draw.line((x1, y, x2, y), fill=GRID, width=2)
        label = f"{value * 100:.0f}%" if percent else (f"{value:.2f}" if ymax <= 1.5 else f"{value:.0f}")
        bbox = draw.textbbox((0, 0), label, font=font(21))
        draw.text((x1 - (bbox[2] - bbox[0]) - 12, y - 13), label, font=font(21), fill=GRAY)


def grouped_bar(filename, chart_title, groups, series_names, values, ymin=0, ymax=None, percent=False, subtitle=None, category="Thực nghiệm", sources=()):
    image, draw = new_image(1900, 1100)
    draw_title(draw, chart_title, subtitle, 1900)
    values_array = np.asarray(values, dtype=float)
    if ymax is None:
        ymax = float(values_array.max() * 1.18) if values_array.size else 1.0
    plot = (155, 155, 1835, 865)
    draw_axes(draw, plot, ymin, ymax, 5, percent)
    x1, y1, x2, y2 = plot
    group_width = (x2 - x1) / len(groups)
    bar_width = min(92, group_width / (len(series_names) + 1))
    for group_idx, group in enumerate(groups):
        center = x1 + group_width * (group_idx + 0.5)
        draw_centered(draw, (center - group_width / 2 + 5, y2 + 12, center + group_width / 2 - 5, y2 + 62), group, 24, True)
        for series_idx, series_name in enumerate(series_names):
            value = float(values_array[series_idx, group_idx])
            left = center - len(series_names) * bar_width / 2 + series_idx * bar_width
            top = y2 - (value - ymin) / (ymax - ymin) * (y2 - y1)
            draw.rectangle((left, top, left + bar_width * 0.78, y2), fill=COLORS[series_idx % len(COLORS)], outline=WHITE)
            label = f"{value * 100:.1f}%" if percent else (f"{value:.3f}" if ymax <= 1.5 else f"{value:.0f}")
            draw_centered(draw, (left - 28, top - 39, left + bar_width + 28, top - 3), label, 19, True)
    legend_width = 300
    total_width = min(1700, legend_width * len(series_names))
    start_x = (1900 - total_width) / 2
    for idx, name in enumerate(series_names):
        x = start_x + idx * (total_width / len(series_names))
        draw.rectangle((x, 984, x + 30, 1014), fill=COLORS[idx % len(COLORS)])
        draw.text((x + 42, 982), name, font=font(23), fill=BLACK)
    return save_image(image, filename, chart_title, category, sources)


def horizontal_grouped_bar(filename, chart_title, categories, series_names, values, xmax=None, subtitle=None, category="Thực nghiệm", sources=()):
    image, draw = new_image(1900, 1300)
    draw_title(draw, chart_title, subtitle, 1900)
    values_array = np.asarray(values, dtype=float)
    xmax = float(xmax or values_array.max() * 1.18 or 1.0)
    plot = (500, 170, 1815, 1140)
    x1, y1, x2, y2 = plot
    draw.line((x1, y1, x1, y2), fill=BLACK, width=3)
    draw.line((x1, y2, x2, y2), fill=BLACK, width=3)
    for tick in range(6):
        value = xmax * tick / 5
        x = x1 + (x2 - x1) * tick / 5
        draw.line((x, y1, x, y2), fill=GRID, width=2)
        draw_centered(draw, (x - 45, y2 + 8, x + 45, y2 + 44), f"{value:.0f}", 20)
    row_height = (y2 - y1) / len(categories)
    bar_height = min(22, row_height / (len(series_names) + 1))
    for category_idx, category_name in enumerate(categories):
        center_y = y1 + row_height * (category_idx + .5)
        draw_centered(draw, (55, center_y - row_height * .46, x1 - 24, center_y + row_height * .46), category_name, 23, True)
        for series_idx, series_name in enumerate(series_names):
            value = float(values_array[series_idx, category_idx])
            top = center_y - len(series_names) * bar_height / 2 + series_idx * bar_height
            right = x1 + value / xmax * (x2 - x1)
            draw.rectangle((x1, top, right, top + bar_height * .78), fill=COLORS[series_idx % len(COLORS)])
            draw.text((right + 8, top - 2), f"{value:.0f}", font=font(17, True), fill=BLACK)
    legend_width = 300
    start_x = (1900 - legend_width * len(series_names)) / 2
    for idx, name in enumerate(series_names):
        x = start_x + idx * legend_width
        draw.rectangle((x, 1215, x + 30, 1245), fill=COLORS[idx % len(COLORS)])
        draw.text((x + 42, 1212), name, font=font(23), fill=BLACK)
    return save_image(image, filename, chart_title, category, sources)


def heatmap(draw, matrix, origin, cell, row_labels, col_labels, value_format="count", max_value=None):
    matrix = np.asarray(matrix, dtype=float)
    ox, oy = origin
    max_value = float(matrix.max()) if max_value is None else float(max_value)
    max_value = max(max_value, 1e-12)
    for col, label in enumerate(col_labels):
        draw_centered(draw, (ox + col * cell, oy - 54, ox + (col + 1) * cell, oy - 5), label, 22, True)
    for row, label in enumerate(row_labels):
        draw_centered(draw, (ox - 112, oy + row * cell, ox - 8, oy + (row + 1) * cell), label, 22, True)
        for col in range(matrix.shape[1]):
            value = float(matrix[row, col])
            ratio = max(0.0, min(1.0, value / max_value))
            shade = int(248 - 150 * ratio)
            fill = (shade, min(255, shade + 8), 255)
            box = (ox + col * cell, oy + row * cell, ox + (col + 1) * cell, oy + (row + 1) * cell)
            draw.rectangle(box, fill=fill, outline=WHITE, width=4)
            label_text = f"{value:.1%}" if value_format == "percent" else (f"{value:.3f}" if value_format == "float" else str(int(round(value))))
            draw_centered(draw, box, label_text, 29, True, NAVY)

## 3. Sơ đồ phương pháp và kiến trúc

In [5]:
def generate_method_diagrams():
    source_model = ROOT / "src" / "models.py"
    source_pipeline = ROOT / "src" / "data_pipeline.py"
    source_train = ROOT / "src" / "train_pipeline.py"
    source_rules = ROOT / "src" / "explainability.py"
    source_db = ROOT / "database" / "schema.sql"

    image, draw = new_image(1900, 1120)
    title_text = "Quy trình hệ thống dự đoán và khuyến nghị"
    draw_title(draw, title_text, "Locked test được cô lập trước hoạt động tối ưu siêu tham số", 1900)
    nodes = [
        (55, "Dữ liệu đầu vào", "Student-Mat\nStudent-Por\nxAPI-Edu-Data"),
        (385, "Tách dữ liệu", "Train pool 80%\nLocked test 20%\nStratified, seed 42"),
        (715, "CV và tiền xử lý", "Feature engineering\nScale/encode\nSMOTE/ADASYN\nFeature selection"),
        (1045, "Tối ưu và huấn luyện", "Optuna 5-fold CV\nEnsemble 5 seeds"),
        (1375, "Đánh giá", "Accuracy, F1-Macro\nPrecision, Recall\nRMSE, R²"),
    ]
    for x, heading, body in nodes:
        draw_box(draw, (x, 180, x + 280, 420), heading, body, SKY, heading_size=29)
    for x, _, _ in nodes[:-1]:
        draw_arrow(draw, (x + 280, 300), (x + 325, 300))
    draw_box(draw, (210, 610, 630, 865), "Giải thích mô hình", "Permutation importance\nXác suất và confidence", PALE)
    draw_box(draw, (740, 610, 1160, 865), "Learning Path Engine", "Risk factors theo luật\nKế hoạch can thiệp 1-4 tuần", PALE_GOLD)
    draw_box(draw, (1270, 610, 1690, 865), "PostgreSQL", "Runs • Predictions\nMetrics • Recommendations", PALE_GREEN)
    draw_arrow(draw, (1515, 420), (1480, 610)); draw_arrow(draw, (1270, 738), (1160, 738)); draw_arrow(draw, (740, 738), (630, 738))
    draw_centered(draw, (170, 930, 1730, 1060), "Đầu ra có thể truy vết: nhãn dự đoán, vector xác suất, độ tin cậy, yếu tố rủi ro và lộ trình học tập", 30, True, NAVY)
    save_image(image, "01_system_pipeline.png", title_text, "Sơ đồ", [source_pipeline, source_train, source_rules, source_db])

    image, draw = new_image(1900, 1120)
    title_text = "Kiến trúc lai CNN-BiLSTM + Context MLP"
    draw_title(draw, title_text, "Hai nhánh xử lý dữ liệu tuần tự và dữ liệu bối cảnh trước khi hợp nhất", 1900)
    draw_box(draw, (55, 170, 330, 370), "Đầu vào tuần tự", "Student: G1, G2\nxAPI: 4 chỉ báo", PALE)
    draw_box(draw, (450, 145, 830, 395), "Nhánh CNN-BiLSTM", "Conv1D + BatchNorm\nReLU + Dropout\nBiLSTM hai chiều", SKY)
    draw_box(draw, (930, 165, 1260, 375), "Attention Pooling", "Trọng số softmax\nTổng có trọng số", SKY)
    draw_box(draw, (55, 655, 330, 855), "Đầu vào bối cảnh", "Biến số\nBiến phân loại", PALE)
    draw_box(draw, (450, 625, 830, 885), "Context MLP", "Min-Max scaling\nLabel encoding\nLinear-ReLU-Dropout\nLinear-ReLU", SKY)
    draw_box(draw, (1370, 390, 1685, 670), "Fusion", "Concatenate\nLinear + ReLU\nDropout", PALE_GOLD)
    draw_box(draw, (1715, 420, 1885, 660), "Đầu ra", "3 logits\nSoftmax\nLow\nMedium\nHigh", PALE_GREEN, heading_size=23, body_size=19)
    draw_arrow(draw, (330, 270), (450, 270)); draw_arrow(draw, (830, 270), (930, 270)); draw_arrow(draw, (330, 755), (450, 755))
    draw_arrow(draw, (1260, 270), (1370, 475)); draw_arrow(draw, (830, 755), (1370, 590)); draw_arrow(draw, (1685, 530), (1730, 530))
    draw_centered(draw, (360, 955, 1540, 1055), "Ensemble: trung bình hóa xác suất của 5 mô hình được huấn luyện với các seed cố định", 30, True, NAVY)
    save_image(image, "02_hybrid_architecture.png", title_text, "Sơ đồ", [source_model, source_pipeline])

    image, draw = new_image(1800, 1050)
    title_text = "Giao thức chống rò rỉ dữ liệu"
    draw_title(draw, title_text, "Mọi trạng thái tiền xử lý được fit trên phần train tương ứng", 1800)
    draw_box(draw, (70, 180, 420, 365), "Dữ liệu gốc", "Tách stratified một lần", PALE)
    draw_box(draw, (560, 120, 970, 350), "Train pool 80%", "Tham gia Optuna\nvà huấn luyện ensemble", SKY)
    draw_box(draw, (1210, 120, 1640, 350), "Locked test 20%", "Không dùng chọn feature,\nsiêu tham số hoặc epoch", PALE_GOLD)
    draw_arrow(draw, (420, 270), (560, 235)); draw_arrow(draw, (420, 270), (1210, 235))
    draw_box(draw, (150, 570, 500, 820), "Train fold", "Fit scaler/encoder\nResampling\nFit feature selector", SKY)
    draw_box(draw, (725, 570, 1075, 820), "Validation fold", "Chỉ transform\nKhông oversampling", PALE_GREEN)
    draw_box(draw, (1300, 570, 1650, 820), "Điểm trial", "F1-Macro của fold\nTrung bình 5 fold", PALE)
    draw_arrow(draw, (765, 350), (380, 570)); draw_arrow(draw, (765, 350), (900, 570)); draw_arrow(draw, (500, 695), (725, 695)); draw_arrow(draw, (1075, 695), (1300, 695))
    draw_centered(draw, (120, 900, 1680, 1005), "Locked test chỉ được mở sau khi kiến trúc và siêu tham số đã cố định", 31, True, RED)
    save_image(image, "03_leakage_control.png", title_text, "Sơ đồ", [source_pipeline, source_train])

    image, draw = new_image(1800, 1050)
    title_text = "Luồng xử lý của Rule-based Learning Path Engine"
    draw_title(draw, title_text, width=1800)
    nodes = [
        (70, "Dự đoán", "Lớp + confidence"), (405, "Feature snapshot", "Điểm, chuyên cần,\ntương tác, hỗ trợ"),
        (740, "Phát hiện rủi ro", "Luật ngưỡng\npriority 1-3"), (1075, "Xếp risk band", "stable / moderate / high"),
        (1410, "Sinh lộ trình", "Mục tiêu và hành động\ntheo tuần"),
    ]
    for x, heading, body in nodes:
        draw_box(draw, (x, 250, x + 280, 500), heading, body, SKY if x < 1000 else PALE_GOLD)
    for x, _, _ in nodes[:-1]:
        draw_arrow(draw, (x + 280, 375), (x + 325, 375))
    phases = [(140, "Tuần 1", "Khôi phục chuyên cần"), (530, "Tuần 1-2", "Bù kiến thức / học liệu"), (920, "Tuần 2-4", "Ổn định tự học / tương tác"), (1310, "Cuối tuần", "Đánh giá và điều chỉnh")]
    for x, heading, body in phases:
        draw_box(draw, (x, 700, x + 350, 885), heading, body, PALE_GREEN, heading_size=30, body_size=24)
    draw_arrow(draw, (1550, 500), (1485, 700))
    save_image(image, "04_rule_engine.png", title_text, "Sơ đồ", [source_rules])

    image, draw = new_image(1800, 1100)
    title_text = "Mô hình lưu trữ PostgreSQL"
    draw_title(draw, title_text, width=1800)
    draw_box(draw, (650, 120, 1150, 330), "paper_runs", "paper_run_id (PK)\ngenerated_at • status\nrun_payload JSONB", SKY)
    draw_box(draw, (70, 500, 500, 810), "paper_predictions", "run_id (FK)\ndataset • split • row_index\ntrue/predicted label\nprobability • confidence\noriginal_features JSONB", PALE)
    draw_box(draw, (685, 500, 1115, 810), "paper_evaluation_metrics", "run_id (FK)\naccuracy • precision_macro\nrecall_macro • f1_macro\nrmse • r2 • payload", PALE_GREEN, heading_size=27)
    draw_box(draw, (1300, 500, 1730, 810), "paper_learning_recommendations", "run_id (FK)\nrisk_band • confidence\nfeature_snapshot JSONB\nrecommended_learning_path JSONB", PALE_GOLD, heading_size=25, body_size=22)
    draw_arrow(draw, (780, 330), (300, 500)); draw_arrow(draw, (900, 330), (900, 500)); draw_arrow(draw, (1020, 330), (1515, 500))
    draw_centered(draw, (180, 900, 1620, 1020), "Mỗi phiên chạy liên kết dự đoán, metric và khuyến nghị để hỗ trợ kiểm toán và tái lập", 31, True, NAVY)
    save_image(image, "05_postgresql_schema.png", title_text, "Sơ đồ", [source_db])


generate_method_diagrams()
print("Đã sinh 5 sơ đồ phương pháp.")

Đã sinh 5 sơ đồ phương pháp.


## 4. Biểu đồ dữ liệu, đánh giá mô hình và khuyến nghị

In [6]:
def all_sources(key):
    return [loaded[dataset]["sources"][key] for dataset in DATASETS]


def generate_dataset_figures():
    groups = [loaded[d]["config"]["label"] for d in DATASETS]
    raw_sizes = [len(loaded[d]["raw"]) for d in DATASETS]
    train_sizes = [len(loaded[d]["train"]) for d in DATASETS]
    test_sizes = [len(loaded[d]["test"]) for d in DATASETS]
    grouped_bar("06_dataset_split_sizes.png", "Quy mô dữ liệu và phân hoạch thực nghiệm", groups,
                ["Dữ liệu gốc", "Train pool", "Locked test"], [raw_sizes, train_sizes, test_sizes],
                ymin=0, ymax=max(raw_sizes) * 1.18, subtitle="Số lượng bản ghi đọc trực tiếp từ CSV",
                sources=all_sources("raw") + all_sources("train") + all_sources("test"))

    train_counts, test_counts = [], []
    for dataset in DATASETS:
        target = loaded[dataset]["config"]["target"]
        train_counts.append([int((pd.to_numeric(loaded[dataset]["train"][target]) == cls).sum()) for cls in range(3)])
        test_counts.append([int((pd.to_numeric(loaded[dataset]["test"][target]) == cls).sum()) for cls in range(3)])
    grouped_bar("07_train_pool_class_counts.png", "Phân bố lớp trong train pool", groups, CLASS_NAMES,
                np.asarray(train_counts).T, ymin=0, subtitle="Số mẫu theo nhãn Low, Medium và High",
                sources=all_sources("train"))
    grouped_bar("08_locked_test_class_counts.png", "Phân bố lớp trong locked test", groups, CLASS_NAMES,
                np.asarray(test_counts).T, ymin=0, subtitle="Số mẫu thật dùng cho đánh giá cuối cùng",
                sources=all_sources("test"))
    percentages = np.asarray(test_counts, dtype=float)
    percentages = (percentages / percentages.sum(axis=1, keepdims=True)).T
    grouped_bar("09_locked_test_class_percentages.png", "Tỷ trọng lớp trong locked test", groups, CLASS_NAMES,
                percentages, ymin=0, ymax=max(0.75, percentages.max() * 1.18), percent=True,
                sources=all_sources("test"))


def generate_evaluation_figures():
    groups = [loaded[d]["config"]["label"] for d in DATASETS]
    metric_names = ["Accuracy", "Precision-Macro", "Recall-Macro", "F1-Macro"]
    metric_values = [[loaded[d]["metrics"][metric] for d in DATASETS] for metric in metric_names]
    grouped_bar("10_locked_test_metrics.png", "Metric phân loại trên locked test", groups, metric_names, metric_values,
                ymin=0.65, ymax=0.96, subtitle="Các giá trị đã được tính lại và đối chiếu với JSON",
                sources=all_sources("metrics") + all_sources("predictions"))
    grouped_bar("11_f1_macro_comparison.png", "So sánh F1-Macro cuối cùng", groups, ["F1-Macro locked test"],
                [[loaded[d]["metrics"]["F1-Macro"] for d in DATASETS]], ymin=0.65, ymax=0.94,
                sources=all_sources("metrics"))

    cv = [loaded[d]["cv_f1"] for d in DATASETS]
    test = [loaded[d]["metrics"]["F1-Macro"] for d in DATASETS]
    grouped_bar("12_cv_vs_locked_test_f1.png", "F1-Macro: Cross-validation và Locked test", groups,
                ["Best CV F1", "Locked-test F1"], [cv, test], ymin=0.70, ymax=0.94,
                subtitle="Chênh lệch phản ánh suy giảm tổng quát hóa ngoài mẫu",
                sources=all_sources("report") + all_sources("metrics"))
    gaps = [cv_value - test_value for cv_value, test_value in zip(cv, test)]
    grouped_bar("13_cv_test_generalization_gap.png", "Chênh lệch F1-Macro giữa CV và Locked test", groups,
                ["CV F1 - Test F1"], [gaps], ymin=0, ymax=max(gaps) * 1.35,
                sources=all_sources("report") + all_sources("metrics"))

    grouped_bar("14_rmse_r2.png", "RMSE và R² trên locked test", groups, ["RMSE", "R²"],
                [[loaded[d]["metrics"]["RMSE"] for d in DATASETS], [loaded[d]["metrics"]["R2"] for d in DATASETS]],
                ymin=0, ymax=0.85, subtitle="RMSE thấp hơn và R² cao hơn là thuận lợi",
                sources=all_sources("metrics") + all_sources("predictions"))

    image, draw = new_image(1900, 1040)
    title_text = "Ma trận nhầm lẫn trên locked test"
    draw_title(draw, title_text, "Hàng: nhãn thật; cột: nhãn dự đoán", 1900)
    for idx, dataset in enumerate(DATASETS):
        ox = 105 + idx * 610
        draw_centered(draw, (ox, 142, ox + 520, 200), loaded[dataset]["config"]["label"], 32, True, NAVY)
        heatmap(draw, loaded[dataset]["cm"], (ox + 105, 275), 140, CLASS_NAMES, CLASS_NAMES, "count")
    save_image(image, "15_confusion_matrices_counts.png", title_text, "Thực nghiệm", all_sources("predictions"))

    image, draw = new_image(1900, 1040)
    title_text = "Ma trận nhầm lẫn chuẩn hóa theo nhãn thật"
    draw_title(draw, title_text, "Mỗi hàng có tổng bằng 100%", 1900)
    for idx, dataset in enumerate(DATASETS):
        cm = loaded[dataset]["cm"].astype(float)
        normalized = cm / cm.sum(axis=1, keepdims=True)
        ox = 105 + idx * 610
        draw_centered(draw, (ox, 142, ox + 520, 200), loaded[dataset]["config"]["label"], 32, True, NAVY)
        heatmap(draw, normalized, (ox + 105, 275), 140, CLASS_NAMES, CLASS_NAMES, "percent", max_value=1.0)
    save_image(image, "16_confusion_matrices_normalized.png", title_text, "Thực nghiệm", all_sources("predictions"))

    for figure_number, metric, title_text in [
        (17, "Precision", "Precision theo từng lớp"),
        (18, "Recall", "Recall theo từng lớp"),
        (19, "F1", "F1-score theo từng lớp"),
    ]:
        values = [[float(loaded[d]["per_class"].set_index("Class").loc[class_name, metric]) for d in DATASETS] for class_name in CLASS_NAMES]
        grouped_bar(f"{figure_number:02d}_per_class_{metric.lower()}.png", title_text, groups, CLASS_NAMES, values,
                    ymin=0.55, ymax=1.01, sources=all_sources("predictions"))


def generate_confidence_figures():
    groups = [loaded[d]["config"]["label"] for d in DATASETS]
    bins = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.000001]
    bin_labels = ["<0.50", "0.50-0.59", "0.60-0.69", "0.70-0.79", "0.80-0.89", "≥0.90"]
    values = []
    for dataset in DATASETS:
        confidence = loaded[dataset]["predictions"]["Confidence"]
        counts = pd.cut(confidence, bins=bins, labels=bin_labels, right=False, include_lowest=True).value_counts().reindex(bin_labels, fill_value=0)
        values.append(counts.values.tolist())
    grouped_bar("20_confidence_histograms.png", "Phân bố độ tin cậy dự đoán", bin_labels, groups, values,
                ymin=0, subtitle="Tần suất confidence của ensemble theo khoảng xác suất",
                sources=all_sources("predictions"))

    correct_means, incorrect_means = [], []
    for dataset in DATASETS:
        frame = loaded[dataset]["predictions"]
        correct = frame["True_Label"] == frame["Pred_Label"]
        correct_means.append(float(frame.loc[correct, "Confidence"].mean()))
        incorrect_means.append(float(frame.loc[~correct, "Confidence"].mean()))
    grouped_bar("21_confidence_correct_vs_incorrect.png", "Độ tin cậy trung bình: dự đoán đúng và sai", groups,
                ["Dự đoán đúng", "Dự đoán sai"], [correct_means, incorrect_means], ymin=0.45, ymax=1.0,
                sources=all_sources("predictions"))

    quantile_names = ["Min", "Q1", "Median", "Q3", "Max"]
    quantile_values = []
    for dataset in DATASETS:
        quantiles = loaded[dataset]["predictions"]["Confidence"].quantile([0, .25, .5, .75, 1]).values
        quantile_values.append(quantiles.tolist())
    grouped_bar("22_confidence_quantiles.png", "Các phân vị của độ tin cậy", quantile_names, groups, quantile_values,
                ymin=0.30, ymax=1.02, sources=all_sources("predictions"))

    true_counts, pred_counts = [], []
    for dataset in DATASETS:
        frame = loaded[dataset]["predictions"]
        true_counts.append([int((frame["True_Label"] == cls).sum()) for cls in range(3)])
        pred_counts.append([int((frame["Pred_Label"] == cls).sum()) for cls in range(3)])
    for idx, class_name in enumerate(CLASS_NAMES):
        pass
    image, draw = new_image(1900, 1100)
    title_text = "Phân bố nhãn thật và nhãn dự đoán"
    draw_title(draw, title_text, "Đối chiếu số lượng theo từng lớp trong locked test", 1900)
    panel_width = 600
    max_count = max(max(row) for row in true_counts + pred_counts)
    for dataset_idx, dataset in enumerate(DATASETS):
        ox = 70 + dataset_idx * 610
        draw_centered(draw, (ox, 145, ox + 520, 200), loaded[dataset]["config"]["label"], 31, True, NAVY)
        plot = (ox + 60, 250, ox + 520, 850)
        draw_axes(draw, plot, 0, max_count * 1.12, 5)
        for cls in range(3):
            center = plot[0] + (cls + .5) * (plot[2] - plot[0]) / 3
            for series_idx, count in enumerate([true_counts[dataset_idx][cls], pred_counts[dataset_idx][cls]]):
                left = center - 54 + series_idx * 58
                top = plot[3] - count / (max_count * 1.12) * (plot[3] - plot[1])
                draw.rectangle((left, top, left + 48, plot[3]), fill=[BLUE, GOLD][series_idx])
                draw_centered(draw, (left - 10, top - 34, left + 58, top), str(count), 18, True)
            draw_centered(draw, (center - 70, plot[3] + 12, center + 70, plot[3] + 52), CLASS_NAMES[cls], 21, True)
    draw.rectangle((720, 985, 750, 1015), fill=BLUE); draw.text((762, 982), "Nhãn thật", font=font(23), fill=BLACK)
    draw.rectangle((980, 985, 1010, 1015), fill=GOLD); draw.text((1022, 982), "Nhãn dự đoán", font=font(23), fill=BLACK)
    save_image(image, "23_true_vs_predicted_distribution.png", title_text, "Thực nghiệm", all_sources("predictions"))

    image, draw = new_image(1900, 1040)
    title_text = "Hồ sơ xác suất dự đoán trung bình theo nhãn thật"
    draw_title(draw, title_text, "Mỗi ô là trung bình Prob_Class_k của các mẫu thuộc một nhãn thật", 1900)
    for idx, dataset in enumerate(DATASETS):
        frame = loaded[dataset]["predictions"]
        matrix = []
        for true_class in range(3):
            subset = frame[frame["True_Label"] == true_class]
            matrix.append([float(subset[f"Prob_Class_{pred_class}"].mean()) for pred_class in range(3)])
        ox = 105 + idx * 610
        draw_centered(draw, (ox, 142, ox + 520, 200), loaded[dataset]["config"]["label"], 32, True, NAVY)
        heatmap(draw, matrix, (ox + 105, 275), 140, CLASS_NAMES, CLASS_NAMES, "percent", max_value=1.0)
    save_image(image, "24_mean_probability_by_true_class.png", title_text, "Thực nghiệm", all_sources("predictions"))


def feature_importance_panel(image, draw, dataset, box, top_n=10, shared_scale=None):
    x1, y1, x2, y2 = box
    frame = loaded[dataset]["importance"].head(top_n).iloc[::-1]
    max_value = max(float(shared_scale or frame["Importance"].clip(lower=0).max()), 1e-12)
    row_height = (y2 - y1) / max(len(frame), 1)
    label_width = min(310, (x2 - x1) * 0.43)
    for idx, row in enumerate(frame.itertuples()):
        y = y1 + idx * row_height
        label = str(row.Feature)
        draw_centered(draw, (x1, y, x1 + label_width - 10, y + row_height), label, 19)
        value = max(float(row.Importance), 0.0)
        length = (x2 - x1 - label_width - 95) * value / max_value
        draw.rectangle((x1 + label_width, y + row_height * .25, x1 + label_width + length, y + row_height * .72), fill=BLUE)
        draw.text((x1 + label_width + length + 8, y + row_height * .27), f"{row.Importance:.4f}", font=font(17), fill=GRAY)


def generate_explainability_figures():
    image, draw = new_image(1900, 1200)
    title_text = "Permutation feature importance"
    draw_title(draw, title_text, "Mức giảm F1-Macro khi hoán vị đặc trưng; hiển thị 8 đặc trưng cao nhất", 1900)
    shared_scale = max(float(loaded[d]["importance"]["Importance"].clip(lower=0).max()) for d in DATASETS)
    for idx, dataset in enumerate(DATASETS):
        ox = 60 + idx * 620
        draw_centered(draw, (ox, 145, ox + 550, 205), loaded[dataset]["config"]["label"], 32, True, NAVY)
        feature_importance_panel(image, draw, dataset, (ox, 225, ox + 560, 1050), top_n=8, shared_scale=shared_scale)
    draw_centered(draw, (120, 1080, 1780, 1160), "Thang đo chung giữa ba bộ dữ liệu; giá trị âm được giữ trong nhãn nhưng thanh được chặn tại 0", 24, False, GRAY)
    save_image(image, "25_feature_importance_combined.png", title_text, "Giải thích", all_sources("importance"))

    for number, dataset in enumerate(DATASETS, start=26):
        image, draw = new_image(1800, 1150)
        title_text = f"Feature importance - {loaded[dataset]['config']['label']}"
        draw_title(draw, title_text, "Top 12 đặc trưng theo permutation importance", 1800)
        feature_importance_panel(image, draw, dataset, (120, 170, 1680, 1050), top_n=12)
        save_image(image, f"{number:02d}_feature_importance_{dataset.replace('-', '_')}.png", title_text, "Giải thích", [loaded[dataset]["sources"]["importance"]])


def parse_json_list(value):
    if pd.isna(value) or value == "":
        return []
    parsed = json.loads(value)
    return parsed if isinstance(parsed, list) else []


def generate_recommendation_figures():
    groups = [loaded[d]["config"]["label"] for d in DATASETS]
    risk_names = ["stable", "moderate", "high"]
    risk_values = []
    for dataset in DATASETS:
        counts = loaded[dataset]["recommendations"]["risk_band"].value_counts()
        risk_values.append([int(counts.get(name, 0)) for name in risk_names])
    grouped_bar("29_risk_band_distribution.png", "Phân bố mức rủi ro của Learning Path Engine", groups,
                risk_names, np.asarray(risk_values).T, ymin=0, sources=all_sources("recommendations"))

    counters = {}
    all_risk_codes = set()
    for dataset in DATASETS:
        counter = Counter()
        for value in loaded[dataset]["recommendations"]["risk_factors"]:
            for factor in parse_json_list(value):
                counter[str(factor.get("code", "unknown"))] += 1
        counters[dataset] = counter
        all_risk_codes.update(counter)
    ordered_codes = sorted(all_risk_codes, key=lambda code: -sum(counters[d][code] for d in DATASETS))
    ordered_codes = ordered_codes[:10]
    values = [[counters[d][code] for code in ordered_codes] for d in DATASETS]
    horizontal_grouped_bar("30_risk_factor_frequency.png", "Tần suất các yếu tố rủi ro", ordered_codes, groups, values,
                           subtitle="Đếm từ trường risk_factors JSON của từng sinh viên",
                           sources=all_sources("recommendations"))

    phase_counters = {}
    all_phases = set()
    for dataset in DATASETS:
        counter = Counter()
        for value in loaded[dataset]["recommendations"]["learning_path"]:
            for step in parse_json_list(value):
                counter[str(step.get("phase", "Không xác định"))] += 1
        phase_counters[dataset] = counter
        all_phases.update(counter)
    ordered_phases = sorted(all_phases, key=lambda phase: -sum(phase_counters[d][phase] for d in DATASETS))
    values = [[phase_counters[d][phase] for phase in ordered_phases] for d in DATASETS]
    grouped_bar("31_learning_path_phase_frequency.png", "Tần suất giai đoạn trong lộ trình học tập", ordered_phases, groups, values,
                ymin=0, subtitle="Đếm từ trường learning_path JSON",
                sources=all_sources("recommendations"))


def generate_hyperparameter_figure():
    fields = [
        "learning_rate", "weight_decay", "batch_size", "oversample_method", "smote_ratio",
        "resampling_k_neighbors", "cnn_channels", "cnn_kernel_size", "lstm_hidden_dim",
        "context_hidden_dim", "fusion_hidden_dim", "dropout", "sequence_dropout",
        "context_dropout", "fusion_dropout",
    ]
    image, draw = new_image(1900, 1250)
    title_text = "Siêu tham số tốt nhất từ Optuna"
    draw_title(draw, title_text, "Giá trị đọc từ báo cáo thực nghiệm của từng bộ dữ liệu", 1900)
    left, top = 80, 165
    col_widths = [470, 430, 430, 430]
    row_height = 62
    headers = ["Siêu tham số"] + [loaded[d]["config"]["label"] for d in DATASETS]
    x_positions = [left]
    for width in col_widths:
        x_positions.append(x_positions[-1] + width)
    for col, header in enumerate(headers):
        draw.rectangle((x_positions[col], top, x_positions[col + 1], top + row_height), fill=NAVY, outline=WHITE, width=2)
        draw_centered(draw, (x_positions[col] + 5, top + 3, x_positions[col + 1] - 5, top + row_height - 3), header, 23, True, WHITE)
    for row_idx, field in enumerate(fields):
        y1 = top + (row_idx + 1) * row_height
        fill = WHITE if row_idx % 2 == 0 else PALE
        values = [field]
        for dataset in DATASETS:
            value = loaded[dataset]["best_params"].get(field, "-")
            if isinstance(value, float):
                value = f"{value:.6g}"
            values.append(str(value))
        for col, value in enumerate(values):
            draw.rectangle((x_positions[col], y1, x_positions[col + 1], y1 + row_height), fill=fill, outline=GRID, width=2)
            draw_centered(draw, (x_positions[col] + 7, y1 + 3, x_positions[col + 1] - 7, y1 + row_height - 3), value, 20, col == 0)
    save_image(image, "32_optuna_best_hyperparameters.png", title_text, "Thực nghiệm", all_sources("report"))


generate_dataset_figures()
generate_evaluation_figures()
generate_confidence_figures()
generate_explainability_figures()
generate_recommendation_figures()
generate_hyperparameter_figure()
print(f"Đã sinh {len(FIGURE_MANIFEST)} hình trước bước tạo contact sheet.")

Đã sinh 32 hình trước bước tạo contact sheet.


## 5. Manifest, contact sheet và ZIP

In [7]:
def make_contact_sheets(paths, columns=4, thumb_size=(430, 270), page_size=12):
    sheets = []
    for page_index in range(0, len(paths), page_size):
        page_paths = paths[page_index:page_index + page_size]
        rows = math.ceil(len(page_paths) / columns)
        width = columns * (thumb_size[0] + 28) + 40
        height = rows * (thumb_size[1] + 78) + 100
        sheet = Image.new("RGB", (width, height), WHITE)
        draw = ImageDraw.Draw(sheet)
        draw_centered(draw, (20, 12, width - 20, 70), f"Contact sheet {page_index // page_size + 1}", 34, True, NAVY)
        for local_index, path in enumerate(page_paths):
            row, col = divmod(local_index, columns)
            x = 25 + col * (thumb_size[0] + 28)
            y = 80 + row * (thumb_size[1] + 78)
            with Image.open(path) as source:
                preview = source.convert("RGB")
                preview.thumbnail(thumb_size, Image.Resampling.LANCZOS)
            px = x + (thumb_size[0] - preview.width) // 2
            py = y + (thumb_size[1] - preview.height) // 2
            sheet.paste(preview, (px, py))
            draw.rectangle((x, y, x + thumb_size[0], y + thumb_size[1]), outline=GRID, width=2)
            draw_centered(draw, (x, y + thumb_size[1] + 5, x + thumb_size[0], y + thumb_size[1] + 67), path.name, 17, True)
        filename = f"00_contact_sheet_{page_index // page_size + 1}.png"
        output = OUT / filename
        sheet.save(output, dpi=(160, 160), optimize=True)
        sheets.append(output)
    return sheets


manifest = pd.DataFrame(FIGURE_MANIFEST).sort_values("filename").reset_index(drop=True)
manifest.to_csv(OUT / "figure_manifest.csv", index=False, encoding="utf-8-sig")
(OUT / "figure_manifest.json").write_text(json.dumps(FIGURE_MANIFEST, ensure_ascii=False, indent=2), encoding="utf-8")

figure_paths = [OUT / name for name in manifest["filename"]]
contact_sheets = make_contact_sheets(figure_paths)

readme = OUT / "README.txt"
readme.write_text(
    "BỘ HÌNH ẢNH LUẬN VĂN TỪ DỮ LIỆU THẬT\n\n"
    f"Số hình chính: {len(figure_paths)}\n"
    f"Số contact sheet: {len(contact_sheets)}\n"
    "Nguồn của từng hình được liệt kê trong figure_manifest.csv và figure_manifest.json.\n"
    "Notebook tái tạo: notebooks/tao_toan_bo_hinh_anh_bao_cao.ipynb\n",
    encoding="utf-8",
)

with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUT.iterdir()):
        if path.is_file():
            archive.write(path, arcname=path.name)

print(f"Hoàn tất: {len(figure_paths)} hình chính, {len(contact_sheets)} contact sheet.")
print("Manifest:", OUT / "figure_manifest.csv")
print("ZIP:", ZIP_PATH)
manifest

Hoàn tất: 32 hình chính, 3 contact sheet.
Manifest: C:\Huflit\kltn\reports\final\figures\notebook_all\figure_manifest.csv
ZIP: C:\Huflit\kltn\reports\final\bo_hinh_anh_luan_van_tu_du_lieu_that.zip


,filename,title,category,data_sources
0,01_system_pipeline.png,Quy trình hệ thống dự đoán và khuyến nghị,Sơ đồ,src\data_pipeline.py | src\train_pipeline.py |...
1,02_hybrid_architecture.png,Kiến trúc lai CNN-BiLSTM + Context MLP,Sơ đồ,src\models.py | src\data_pipeline.py
2,03_leakage_control.png,Giao thức chống rò rỉ dữ liệu,Sơ đồ,src\data_pipeline.py | src\train_pipeline.py
3,04_rule_engine.png,Luồng xử lý của Rule-based Learning Path Engine,Sơ đồ,src\explainability.py
4,05_postgresql_schema.png,Mô hình lưu trữ PostgreSQL,Sơ đồ,database\schema.sql
5,06_dataset_split_sizes.png,Quy mô dữ liệu và phân hoạch thực nghiệm,Thực nghiệm,data\raw\student-mat.csv | data\raw\student-po...
6,07_train_pool_class_counts.png,Phân bố lớp trong train pool,Thực nghiệm,data\processed\final\student-mat_3class_train_...
7,08_locked_test_class_counts.png,Phân bố lớp trong locked test,Thực nghiệm,data\processed\final\student-mat_3class_locked...
8,09_locked_test_class_percentages.png,Tỷ trọng lớp trong locked test,Thực nghiệm,data\processed\final\student-mat_3class_locked...
9,10_locked_test_metrics.png,Metric phân loại trên locked test,Thực nghiệm,reports\final\metrics\student-mat_3class_locke...
